# Machine Learning — Lab 5
## Gradient Descent, Polynomial Models, and Regularization

**Main Course Learning Outcomes — CLO3, CLO5**

- **CLO3:** Construct and analyze machine learning models by applying algorithmic principles and internal computations.
- **CLO5:** Evaluate machine learning models using appropriate metrics and justify decisions based on performance trade-offs and real-world context.

**Environment:** Python 3 / Jupyter Notebook  
**Libraries:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`

> **Assessment principle:** Full credit requires more than obtaining a low error. You must explain **how the parameters change, why learning rate matters, when flexibility becomes overfitting, and how regularization changes the bias–variance trade-off**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. Gradient-descent mechanics | 30 min | Implement linear-regression loss and parameter updates |
| 2. Learning-rate experiment | 15 min | Compare slow, useful, and unstable step sizes |
| 3. Polynomial regression | 25 min | Increase model flexibility and observe train/validation behavior |
| 4. Ridge and Lasso | 25 min | Control complexity with regularization |
| 5. Model selection & final test | 15 min | Select using validation evidence and evaluate once |
| 6. Debugging, challenge & viva | 10 min | Diagnose update and evaluation errors |
| **Total** | **120 min** | |

### Main idea

This lab connects four ideas:

$$
\boxed{
\text{Loss}
\rightarrow
\text{Optimization}
\rightarrow
\text{Flexibility}
\rightarrow
\text{Regularization}
}
$$

## Learning Objectives

By the end of this lab, you should be able to:

1. compute predictions for a simple linear model;
2. compute Mean Squared Error manually;
3. derive and implement gradients for $w$ and $b$;
4. perform iterative gradient-descent updates;
5. explain the effect of the learning rate;
6. compare batch gradient descent with the idea of mini-batch updates;
7. create polynomial features;
8. diagnose underfitting and overfitting from train/validation error;
9. apply Ridge and Lasso regularization;
10. select model complexity and regularization using validation data.

# Part I — Setup

We will use two datasets in this lab.

### A. Tiny linear dataset

Used for **manual gradient-descent calculations**.

### B. Nonlinear energy-demand dataset

Used to study:

- polynomial model flexibility;
- overfitting;
- Ridge regularization;
- Lasso regularization;
- validation-based model selection.

The target is daily electricity demand.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Machine Learning Lab 5 environment ready.")

# Part II — Gradient Descent from First Principles

For simple linear regression:

$$
\hat{y}=wx+b.
$$

We use Mean Squared Error:

$$
MSE=
\frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i-y_i)^2.
$$

The gradients are:

$$
\frac{\partial MSE}{\partial w}
=
\frac{2}{n}\sum_{i=1}^{n}(\hat{y}_i-y_i)x_i
$$

and

$$
\frac{\partial MSE}{\partial b}
=
\frac{2}{n}\sum_{i=1}^{n}(\hat{y}_i-y_i).
$$

Gradient descent updates:

$$
w\leftarrow w-\alpha\frac{\partial MSE}{\partial w}
$$

$$
b\leftarrow b-\alpha\frac{\partial MSE}{\partial b}.
$$

In [ ]:
# Tiny standardized dataset for transparent gradient calculations.
x_small = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
y_small = np.array([-3.8, -1.9, 0.2, 2.2, 4.1])

tiny_df = pd.DataFrame({
    "x": x_small,
    "y": y_small
})

display(tiny_df)

## Task 2.1 — Predict Before Computing

Start with:

$$
w=0,\qquad b=0.
$$

Before running any code, answer:

1. What prediction will the model make for every point?
2. Will the initial MSE be zero, small, or relatively large?
3. From the data pattern, should the learned slope eventually become positive or negative?
4. Should the intercept stay near zero or become very large?

**Your prediction:**

## Task 2.2 — Implement Prediction and MSE

Complete the two functions.

In [ ]:
def predict_line(x, w, b):
    # TODO: return wx + b for every x.
    prediction = None
    return prediction


def mse_loss(y_true, y_pred):
    # TODO: implement mean squared error manually.
    loss = None
    return loss

In [ ]:
# Self-check after completing the functions.
assert np.allclose(
    predict_line(np.array([0.0, 1.0, 2.0]), 2.0, 1.0),
    np.array([1.0, 3.0, 5.0])
)

assert abs(
    mse_loss(np.array([1.0, 2.0]), np.array([2.0, 4.0])) - 2.5
) < 1e-12

print("Prediction and MSE function tests passed.")

In [ ]:
w0 = 0.0
b0 = 0.0

initial_pred = predict_line(x_small, w0, b0)
initial_loss = mse_loss(y_small, initial_pred)

print("Initial predictions:", initial_pred)
print("Initial MSE:", round(initial_loss, 4))

## Task 2.3 — Implement the Gradients

Complete the gradient function using:

$$
\frac{\partial MSE}{\partial w}
=
\frac{2}{n}\sum_i(\hat{y}_i-y_i)x_i
$$

and

$$
\frac{\partial MSE}{\partial b}
=
\frac{2}{n}\sum_i(\hat{y}_i-y_i).
$$

In [ ]:
def linear_gradients(x, y, w, b):
    y_pred = predict_line(x, w, b)
    error = y_pred - y
    n = len(x)

    # TODO: compute dw and db.
    dw = None
    db = None

    return dw, db

In [ ]:
# Self-check.
x_check = np.array([1.0, 2.0])
y_check = np.array([3.0, 5.0])

dw_check, db_check = linear_gradients(
    x_check, y_check, w=0.0, b=0.0
)

# With predictions [0,0], errors are [-3,-5].
# dw = (2/2) * [(-3)*1 + (-5)*2] = -13
# db = (2/2) * (-8) = -8
assert abs(dw_check - (-13.0)) < 1e-12
assert abs(db_check - (-8.0)) < 1e-12

print("Gradient function tests passed.")

## Task 2.4 — Manual First Update

Using the tiny dataset and initial values:

$$
w=0,\qquad b=0,\qquad \alpha=0.05,
$$

calculate manually:

1. the five predictions;
2. the error vector $\hat{y}-y$;
3. $\frac{\partial MSE}{\partial w}$;
4. $\frac{\partial MSE}{\partial b}$;
5. the updated $w$;
6. the updated $b$.

Do this **before** running the verification cell.

**Your calculation:**

In [ ]:
alpha_demo = 0.05

dw0, db0 = linear_gradients(
    x_small, y_small, w0, b0
)

w1 = w0 - alpha_demo * dw0
b1 = b0 - alpha_demo * db0

loss_after_one = mse_loss(
    y_small,
    predict_line(x_small, w1, b1)
)

print(f"dw = {dw0:.4f}")
print(f"db = {db0:.4f}")
print(f"Updated w = {w1:.4f}")
print(f"Updated b = {b1:.4f}")
print(f"MSE after one update = {loss_after_one:.4f}")

## Task 2.5 — Interpret the Update

Answer:

1. Did the slope move in the direction you predicted?
2. Did the loss decrease after one update?
3. If the gradient for $w$ is negative, why does subtracting the gradient increase $w$?
4. What would happen if the update used `+ alpha * gradient` instead?

## Task 2.6 — Complete Batch Gradient Descent

Complete the training function.

It should:

1. start from supplied $w$ and $b$;
2. compute gradients from the **entire dataset** each epoch;
3. update the parameters;
4. store the MSE after each epoch.

In [ ]:
def batch_gradient_descent(
    x,
    y,
    learning_rate=0.05,
    epochs=100,
    w_start=0.0,
    b_start=0.0
):
    w = float(w_start)
    b = float(b_start)
    losses = []

    for epoch in range(epochs):
        # TODO: compute gradients.
        dw, db = None, None

        # TODO: update w and b.
        # w = ...
        # b = ...

        # TODO: compute and store current loss.
        current_loss = None
        losses.append(current_loss)

    return w, b, np.array(losses)

In [ ]:
# Self-check after implementation.
w_fit, b_fit, losses_fit = batch_gradient_descent(
    x_small,
    y_small,
    learning_rate=0.05,
    epochs=150
)

assert np.isfinite(w_fit)
assert np.isfinite(b_fit)
assert np.all(np.isfinite(losses_fit))
assert losses_fit[-1] < losses_fit[0]

print("Gradient-descent training test passed.")
print("Final w:", round(w_fit, 4))
print("Final b:", round(b_fit, 4))
print("Final MSE:", round(losses_fit[-1], 6))

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(losses_fit)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Batch Gradient Descent — Loss Curve")
plt.show()

## Task 2.7 — Explain the Loss Curve

Answer:

1. Does loss decrease monotonically in this experiment?
2. Does it decrease quickly at the beginning or near the end?
3. What does a nearly flat curve near the end suggest?
4. Is reaching exactly zero MSE necessary for a useful model?

# Part III — Learning Rate Experiment

The learning rate $\alpha$ controls the step size.

A small learning rate can make training very slow.

A very large learning rate can make training unstable or divergent.

## Task 3.1 — Personalized Learning Rate

Enter the last four digits of your student ID.

Your ID assigns one additional learning rate to investigate.

In [ ]:
# TODO: Replace None with the last four digits of your own student ID.
STUDENT_ID_LAST4 = None

if STUDENT_ID_LAST4 is None:
    raise ValueError("Enter the last four digits of your student ID.")

if not isinstance(STUDENT_ID_LAST4, int):
    raise TypeError("STUDENT_ID_LAST4 must be an integer.")

SEED = 5000 + (STUDENT_ID_LAST4 % 5000)

personal_rates = [0.005, 0.02, 0.08, 0.20]
PERSONAL_RATE = personal_rates[SEED % len(personal_rates)]

print("Your experiment seed:", SEED)
print("Your assigned learning rate:", PERSONAL_RATE)

## Task 3.2 — Predict Before Running

We will compare:

$$
\alpha=0.001,\quad 0.05,\quad 0.5,\quad \text{your assigned rate}.
$$

Before running the experiment, predict:

- which will learn slowly;
- which will learn efficiently;
- which may become unstable;
- what you expect for your assigned rate.

**Your prediction:**

In [ ]:
rates = [0.001, 0.05, 0.5, PERSONAL_RATE]
rate_results = []

plt.figure(figsize=(8, 5))

for rate in rates:
    w_r, b_r, losses_r = batch_gradient_descent(
        x_small,
        y_small,
        learning_rate=rate,
        epochs=80
    )

    finite_losses = losses_r[np.isfinite(losses_r)]

    rate_results.append({
        "learning_rate": rate,
        "final_w": w_r,
        "final_b": b_r,
        "final_loss": losses_r[-1],
        "minimum_loss": np.min(finite_losses) if len(finite_losses) else np.nan,
    })

    plt.plot(losses_r, label=f"alpha={rate}")

plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("MSE (log scale)")
plt.title("Learning Rate Comparison")
plt.legend()
plt.show()

display(pd.DataFrame(rate_results))

## Task 3.3 — Analyze Learning Rate Behavior

Answer:

1. Which learning rate converged most slowly?
2. Which reached a low loss efficiently?
3. Did any learning rate oscillate or diverge?
4. How did your assigned learning rate behave?
5. Why is the learning rate a **hyperparameter** rather than a learned model parameter?

# Part IV — Nonlinear Energy-Demand Dataset

Electricity demand often has a nonlinear relationship with temperature:

- very cold days may require heating;
- mild days may require less energy;
- very hot days may require cooling.

This creates a curved relationship that a simple straight line may underfit.

In [ ]:
rng = np.random.default_rng(7411)
n = 230

temperature = rng.uniform(5, 42, size=n)

# Nonlinear demand curve with noise.
demand = (
    520
    + 1.15 * (temperature - 23) ** 2
    + 0.025 * (temperature - 23) ** 3
    + rng.normal(0, 38, size=n)
)

# Add a small number of unusual high-demand days.
high_idx = rng.choice(np.arange(n), size=8, replace=False)
demand[high_idx] += rng.normal(120, 25, size=len(high_idx))

energy_master = pd.DataFrame({
    "temperature_c": np.round(temperature, 2),
    "energy_demand_mwh": np.round(demand, 2),
})

energy = energy_master.sample(
    n=180,
    random_state=SEED,
    replace=False
).reset_index(drop=True)

display(energy.head())
print("Working nonlinear dataset shape:", energy.shape)

In [ ]:
X_energy = energy[["temperature_c"]]
y_energy = energy["energy_demand_mwh"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X_energy,
    y_energy,
    test_size=0.40,
    random_state=SEED
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED
)

print("Train:", X_train.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(
    X_train["temperature_c"],
    y_train,
    alpha=0.7
)
plt.xlabel("Temperature (°C)")
plt.ylabel("Energy Demand (MWh)")
plt.title("Training Data — Temperature vs. Energy Demand")
plt.show()

## Task 4.1 — Predict the Model Shape

Before fitting anything:

1. Does a straight line seem appropriate?
2. What general shape do you observe?
3. What would a degree-2 polynomial add?
4. What risk appears if the polynomial degree becomes very high?

# Part V — Polynomial Regression

Polynomial regression expands one feature into:

$$
x,\;x^2,\;x^3,\ldots,x^d.
$$

Then ordinary linear regression learns coefficients for these transformed features.

The model remains linear in its parameters, but it becomes nonlinear in the original input $x$.

## Task 5.1 — Inspect Polynomial Features

For degree 3, complete the transformation and inspect the first rows.

In [ ]:
poly3 = PolynomialFeatures(
    degree=3,
    include_bias=False
)

# TODO: fit the transformer on training temperature and transform it.
X_train_poly3 = None

if X_train_poly3 is not None:
    names3 = poly3.get_feature_names_out(["temperature_c"])
    display(pd.DataFrame(
        X_train_poly3[:5],
        columns=names3
    ))

## Task 5.2 — Fit Multiple Polynomial Degrees

We will compare degrees:

$$
1,\;2,\;3,\;5,\;12.
$$

For each degree:

- fit on training data;
- calculate training RMSE;
- calculate validation RMSE.

Do **not** use the test set for choosing the degree.

In [ ]:
degrees = [1, 2, 3, 5, 12]
degree_results = []
degree_models = {}

for degree in degrees:
    model = Pipeline(steps=[
        ("poly", PolynomialFeatures(
            degree=degree,
            include_bias=False
        )),
        ("linear", LinearRegression()),
    ])

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    valid_pred = model.predict(X_valid)

    train_rmse = np.sqrt(
        mean_squared_error(y_train, train_pred)
    )
    valid_rmse = np.sqrt(
        mean_squared_error(y_valid, valid_pred)
    )

    degree_results.append({
        "degree": degree,
        "train_rmse": train_rmse,
        "validation_rmse": valid_rmse,
    })

    degree_models[degree] = model

degree_table = pd.DataFrame(degree_results)
display(degree_table.round(3))

## Task 5.3 — Diagnose Underfitting and Overfitting

From the table:

1. Which degree appears to underfit?
2. Which degree has the smallest training RMSE?
3. Which degree has the smallest validation RMSE?
4. Does the model with the smallest training RMSE also have the best validation RMSE?
5. Which degree would you select **before regularization**?
6. Why is validation error the correct selection criterion?

In [ ]:
x_grid = pd.DataFrame({
    "temperature_c": np.linspace(
        X_train["temperature_c"].min(),
        X_train["temperature_c"].max(),
        300
    )
})

for degree in [1, 3, 12]:
    plt.figure(figsize=(7, 5))
    plt.scatter(
        X_train["temperature_c"],
        y_train,
        alpha=0.55,
        label="Training data"
    )
    plt.plot(
        x_grid["temperature_c"],
        degree_models[degree].predict(x_grid),
        linewidth=2,
        label=f"Degree {degree}"
    )
    plt.xlabel("Temperature (°C)")
    plt.ylabel("Energy Demand (MWh)")
    plt.title(f"Polynomial Regression — Degree {degree}")
    plt.legend()
    plt.show()

## Task 5.4 — Visual Interpretation

For the degree 1, degree 3, and degree 12 plots:

1. Which is too rigid?
2. Which follows the broad pattern?
3. Does the degree-12 curve show unnecessary bending?
4. Why can a high-degree model have lower training error but worse validation performance?

# Part VI — Ridge Regularization

Ridge regression minimizes:

$$
MSE
+
\lambda\sum_j w_j^2.
$$

The penalty discourages very large coefficients.

We will apply Ridge to the degree-12 polynomial model.

## Task 6.1 — Predict the Effect of $\lambda$

Before running the experiment, predict what happens as $\lambda$ increases:

- coefficient magnitudes;
- training error;
- validation error;
- model flexibility.

**Your prediction:**

In [ ]:
alphas = [0.0, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
ridge_results = []
ridge_models = {}

for alpha in alphas:
    if alpha == 0.0:
        regressor = LinearRegression()
    else:
        regressor = Ridge(alpha=alpha)

    model = Pipeline(steps=[
        ("poly", PolynomialFeatures(
            degree=12,
            include_bias=False
        )),
        ("scale", StandardScaler()),
        ("regressor", regressor),
    ])

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    valid_pred = model.predict(X_valid)

    ridge_results.append({
        "alpha": alpha,
        "train_rmse": np.sqrt(
            mean_squared_error(y_train, train_pred)
        ),
        "validation_rmse": np.sqrt(
            mean_squared_error(y_valid, valid_pred)
        ),
    })

    ridge_models[alpha] = model

ridge_table = pd.DataFrame(ridge_results)
display(ridge_table.round(3))

## Task 6.2 — Select Ridge Strength

Answer:

1. Which $\alpha$ has the lowest validation RMSE?
2. What happens to training RMSE as regularization becomes stronger?
3. Does moderate regularization improve validation performance?
4. What happens when regularization becomes too strong?
5. Why must $\alpha$ be selected using validation data?

## Task 6.3 — Inspect Coefficient Shrinkage

Compare coefficient magnitudes for:

- no regularization;
- a moderate Ridge value;
- a strong Ridge value.

In [ ]:
def get_regression_coefficients(model):
    reg = model.named_steps["regressor"]
    return np.asarray(reg.coef_, dtype=float).ravel()

chosen_for_inspection = [0.0, 1.0, 100.0]

coef_rows = []

for alpha in chosen_for_inspection:
    coefs = get_regression_coefficients(ridge_models[alpha])
    coef_rows.append({
        "alpha": alpha,
        "L1_sum_abs_coefficients": np.sum(np.abs(coefs)),
        "L2_norm_coefficients": np.sqrt(np.sum(coefs ** 2)),
        "max_abs_coefficient": np.max(np.abs(coefs)),
    })

display(pd.DataFrame(coef_rows).round(4))

## Task 6.4 — Interpret Shrinkage

Answer:

1. Does stronger Ridge reduce the overall coefficient magnitude?
2. Why can smaller coefficients improve generalization?
3. Does Ridge normally set many coefficients exactly to zero?
4. Why was scaling placed before Ridge in the pipeline?

# Part VII — Lasso Regularization

Lasso minimizes:

$$
MSE
+
\lambda\sum_j |w_j|.
$$

Unlike Ridge, Lasso can drive some coefficients exactly to zero.

In [ ]:
lasso_alphas = [0.001, 0.01, 0.1, 1.0, 5.0]
lasso_results = []
lasso_models = {}

for alpha in lasso_alphas:
    model = Pipeline(steps=[
        ("poly", PolynomialFeatures(
            degree=12,
            include_bias=False
        )),
        ("scale", StandardScaler()),
        ("lasso", Lasso(
            alpha=alpha,
            max_iter=50000
        )),
    ])

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    valid_pred = model.predict(X_valid)

    coefs = model.named_steps["lasso"].coef_

    lasso_results.append({
        "alpha": alpha,
        "train_rmse": np.sqrt(
            mean_squared_error(y_train, train_pred)
        ),
        "validation_rmse": np.sqrt(
            mean_squared_error(y_valid, valid_pred)
        ),
        "nonzero_coefficients": int(
            np.count_nonzero(np.abs(coefs) > 1e-10)
        ),
    })

    lasso_models[alpha] = model

lasso_table = pd.DataFrame(lasso_results)
display(lasso_table.round(3))

## Task 7.1 — Ridge vs. Lasso

Use the Ridge and Lasso tables to answer:

1. Which method tends to keep all coefficients but shrink them?
2. Which method can remove polynomial terms by setting coefficients to zero?
3. At what Lasso $\alpha$ do you first observe fewer nonzero coefficients?
4. Does sparsity automatically mean better validation performance?
5. Which method would be attractive if feature selection were important?

# Part VIII — Validation-Based Model Selection

We now compare candidate model families.

The test set remains untouched.

Candidate families:

- degree-1 linear regression;
- best unregularized polynomial degree;
- best degree-12 Ridge model;
- best degree-12 Lasso model.

In [ ]:
best_degree_row = degree_table.loc[
    degree_table["validation_rmse"].idxmin()
]
best_degree = int(best_degree_row["degree"])

best_ridge_row = ridge_table.loc[
    ridge_table["validation_rmse"].idxmin()
]
best_ridge_alpha = float(best_ridge_row["alpha"])

best_lasso_row = lasso_table.loc[
    lasso_table["validation_rmse"].idxmin()
]
best_lasso_alpha = float(best_lasso_row["alpha"])

selection_table = pd.DataFrame([
    {
        "candidate": "Degree 1",
        "validation_rmse": degree_table.loc[
            degree_table["degree"] == 1,
            "validation_rmse"
        ].iloc[0]
    },
    {
        "candidate": f"Best unregularized degree={best_degree}",
        "validation_rmse": best_degree_row["validation_rmse"]
    },
    {
        "candidate": f"Degree 12 Ridge alpha={best_ridge_alpha}",
        "validation_rmse": best_ridge_row["validation_rmse"]
    },
    {
        "candidate": f"Degree 12 Lasso alpha={best_lasso_alpha}",
        "validation_rmse": best_lasso_row["validation_rmse"]
    },
])

display(
    selection_table
    .sort_values("validation_rmse")
    .round(3)
)

## Task 8.1 — Choose the Final Model

Using **validation RMSE only**, state which candidate you would select.

Then justify:

1. why you did not choose by training RMSE;
2. why you did not inspect test RMSE first;
3. whether a much more complex model is worthwhile if validation performance is almost identical;
4. how interpretability and computational simplicity may affect the final choice.

# Part IX — Final Test Evaluation

Only after model selection should the final test set be used.

The code below automatically selects the validation winner from the candidate models.

In [ ]:
candidate_models = {
    "Degree 1": degree_models[1],
    f"Best unregularized degree={best_degree}": degree_models[best_degree],
    f"Degree 12 Ridge alpha={best_ridge_alpha}": ridge_models[best_ridge_alpha],
    f"Degree 12 Lasso alpha={best_lasso_alpha}": lasso_models[best_lasso_alpha],
}

winner_name = selection_table.loc[
    selection_table["validation_rmse"].idxmin(),
    "candidate"
]

winner_model = candidate_models[winner_name]

test_pred = winner_model.predict(X_test)

test_mae = mean_absolute_error(y_test, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
test_r2 = r2_score(y_test, test_pred)

print("Selected model:", winner_name)
print(f"Final Test MAE:  {test_mae:.3f}")
print(f"Final Test RMSE: {test_rmse:.3f}")
print(f"Final Test R^2:  {test_r2:.3f}")

## Task 9.1 — Final Generalization Statement

Write 3–5 sentences that include:

- the selected model;
- why it was selected;
- final test RMSE;
- final test $R^2$;
- whether the test result is reasonably consistent with validation performance;
- one limitation of this experiment.

Do not tune the model again after seeing the test result.

# Part X — Deliberate Debugging

A student writes:

```python
w = w + learning_rate * dw
b = b + learning_rate * db
```

even though `dw` and `db` are ordinary gradients.

Explain the error.

## Task 10.1 — Gradient Sign Bug

Answer:

1. In which direction does the gradient point?
2. Why does gradient descent subtract it?
3. What may happen to the loss if we repeatedly add the gradient?
4. Is it possible for a few updates to appear harmless before divergence becomes obvious?

## Task 10.2 — Correct the Update

Complete:

In [ ]:
def one_correct_update(x, y, w, b, learning_rate):
    dw, db = linear_gradients(x, y, w, b)

    # TODO: perform one correct gradient-descent update.
    w_new = None
    b_new = None

    return w_new, b_new

In [ ]:
w_test, b_test = one_correct_update(
    x_small,
    y_small,
    w=0.0,
    b=0.0,
    learning_rate=0.05
)

assert abs(w_test - w1) < 1e-12
assert abs(b_test - b1) < 1e-12

print("Gradient update debugging test passed.")

# Part XI — Personalized Complexity Challenge

Your student-ID seed assigns one polynomial degree.

You will compare it with the validation-selected unregularized degree.

In [ ]:
challenge_degrees = [2, 4, 6, 8, 10, 14]
PERSONAL_DEGREE = challenge_degrees[
    SEED % len(challenge_degrees)
]

print("Your assigned polynomial degree:", PERSONAL_DEGREE)

## Task 11.1 — Predict Before Fitting

Before running the model, predict:

1. whether your degree is more or less flexible than the selected unregularized degree;
2. whether its training RMSE should be lower or higher;
3. whether its validation RMSE is guaranteed to improve;
4. whether regularization may help if your degree is high.

**Your prediction:**

In [ ]:
personal_model = Pipeline(steps=[
    ("poly", PolynomialFeatures(
        degree=PERSONAL_DEGREE,
        include_bias=False
    )),
    ("linear", LinearRegression()),
])

personal_model.fit(X_train, y_train)

personal_train_pred = personal_model.predict(X_train)
personal_valid_pred = personal_model.predict(X_valid)

personal_train_rmse = np.sqrt(
    mean_squared_error(y_train, personal_train_pred)
)
personal_valid_rmse = np.sqrt(
    mean_squared_error(y_valid, personal_valid_pred)
)

print("Assigned degree:", PERSONAL_DEGREE)
print(f"Training RMSE:   {personal_train_rmse:.3f}")
print(f"Validation RMSE: {personal_valid_rmse:.3f}")

## Task 11.2 — Analyze Your Result

Answer:

1. Was your prediction correct?
2. Did extra flexibility help training performance?
3. Did it help validation performance?
4. Does your result illustrate underfitting, useful flexibility, or overfitting?
5. Would you select this model? Justify using validation evidence.

# Individual Understanding Check

Your instructor may select one question for a 60–90 second explanation.

1. Explain one gradient-descent update for $w$ and $b$.
2. Why can a very large learning rate make loss increase?
3. Why can degree-12 polynomial regression overfit?
4. What is the difference between training error and validation error?
5. How does Ridge regularization change coefficients?
6. How does Lasso differ from Ridge?
7. Why is scaling important before regularization?
8. For your personalized degree, explain the train/validation behavior you observed.

You should be able to answer without reading a prepared paragraph.

# Reflection

Answer concisely in your own words.

1. What did implementing gradient descent manually help you understand?
2. Which learning-rate behavior was most important to observe?
3. Why is the lowest training error not the main goal?
4. What problem does regularization address?
5. When would you prefer a simpler model even if a more complex model performs slightly better?

**Your reflection:**

# Submission Checklist

Before submitting, confirm that your notebook contains:

- [ ] prediction and MSE functions;
- [ ] gradient function;
- [ ] manual first-update calculation;
- [ ] completed batch gradient descent;
- [ ] learning-rate experiment and interpretation;
- [ ] your own student-ID-derived experiment;
- [ ] polynomial-feature transformation;
- [ ] train/validation comparison across polynomial degrees;
- [ ] visual comparison of degree 1, 3, and 12;
- [ ] Ridge experiment;
- [ ] Lasso experiment;
- [ ] Ridge vs. Lasso interpretation;
- [ ] validation-based model selection;
- [ ] final test evaluation;
- [ ] corrected gradient-sign debugging task;
- [ ] personalized complexity challenge;
- [ ] reflection answers;
- [ ] all required code cells executed successfully.

# Assessment — 10 Marks

| Component | Marks |
|---|---:|
| Correct implementation | **2** |
| Algorithmic / modeling justification | **3** |
| Experimental analysis | **2** |
| Trace / prediction / debugging | **1** |
| Individual understanding check | **1** |
| Code quality and submission completeness | **1** |
| **Total** | **10** |

### Marking emphasis

Full marks require showing that you understand:

$$
\boxed{
\text{gradient}
\rightarrow
\text{update}
\rightarrow
\text{loss change}
\rightarrow
\text{model flexibility}
\rightarrow
\text{generalization}
}
$$

# Lab 5 Summary

You should now be able to connect optimization and generalization.

### Gradient descent

$$
\theta
\leftarrow
\theta-\alpha\nabla_\theta L
$$

- gradients indicate how loss changes;
- learning rate controls update size;
- loss curves reveal optimization behavior.

### Polynomial regression

- higher degree increases flexibility;
- training error usually decreases as flexibility increases;
- validation error may eventually increase due to overfitting.

### Regularization

Ridge:

$$
MSE+\lambda\sum_j w_j^2
$$

Lasso:

$$
MSE+\lambda\sum_j |w_j|
$$

- Ridge shrinks coefficients;
- Lasso can set some coefficients to zero;
- regularization strength must be selected using validation evidence.

**Next lab:** Logistic Regression — Forward Pass, Loss, and Learning.